# An SMA150 filter under two crisis families

This notebook mirrors `spy_sma_ai_bubble_us_bankruptcy.py`. It stresses a four-asset SPY / TLT / GLD / BTC-USD panel with a 150-bar moving-average filter across the `ai_bubble` family and the `us_bankruptcy` 2x2 (dollar direction x aftermath).

**Every result below is conditional on its scenario occurring. These are what-ifs, not forecasts, and no probability is attached to any scenario.**

Needs `pip install "crisispaths[data,report]"` and network access (Yahoo Finance, FRED). Outputs are intentionally not saved in this file.

## 1. The data

SPY sets the business-day calendar; BTC-USD (which trades on weekends) is sampled on it. GLD (2004) and BTC-USD (2014) are NaN before their first price.

In [ ]:
import pandas as pd
import yfinance as yf

import crisispaths as cp
from crisispaths.data import YFinanceFredSource
from crisispaths.report import write_report
from crisispaths.runner import run_sweep

TICKERS = ["SPY", "TLT", "GLD", "BTC-USD"]
raw = yf.download(TICKERS, start="1999-01-01", progress=False, auto_adjust=True)["Close"]
prices = raw[TICKERS].astype(float)
prices = prices.loc[prices["SPY"].dropna().index]
prices.tail()

## 2. The strategy

A strategy maps prices to target weights on the same index; NaN means zero and the rest is cash. The harness shifts the weights by one bar, so a signal never trades on the bar that produced it, and `crisispaths` audits the function for look-ahead before running.

In [ ]:
def sma150(p: pd.DataFrame) -> pd.DataFrame:
    ma = p.rolling(150).mean()
    return (p > ma).astype(float).where(ma.notna()) / p.shape[1]

## 3. What scenarios exist

Each family is a base case plus independent axes. `describe_axes` lists them with their defaults.

In [ ]:
cp.describe_axes("us_bankruptcy")["axes"]["usd"]["options"]

## 4. Run the sweeps

Costs are not zero in a crisis: 5 bps per unit of turnover times a crisis multiplier of 3. `n_paths=200` is the minimum for tail statistics; use more for anything you rely on. Results are written to `out_dir` and a killed run resumes from the rows already on disk.

In [ ]:
clock = cp.Clock(calendar=prices.index)
source = YFinanceFredSource()
common = dict(n_paths=200, seed=0, clock=clock, source=source, workers=1,
              costs_bps=5.0, cost_multiplier=3.0)

ai = run_sweep(prices, sma150, "ai_bubble", {"complexity": "standard"},
               out_dir="out_spy/ai_bubble", **common)
us = run_sweep(prices, sma150, "us_bankruptcy",
               {"complexity": "full", "usd": ["debasement", "haven"],
                "aftermath": ["prolonged_bear", "v_recovery"]},
               out_dir="out_spy/us_bankruptcy", **common)

## 5. Read the results scenario by scenario

Each row is one scenario's own result. Do not average the rows: the differences between them are how much the answer depends on judgment calls, not an expected return.

In [ ]:
def headline(t):
    return t.groupby("scenario", sort=False).agg(
        median_wealth=("terminal_wealth", "median"),
        median_benchmark=("bench_terminal_wealth", "median"),
        p_ruin=("ruined", "mean"),
        median_max_drawdown=("max_drawdown_from_peak", "median"),
        share_recovered=("recovered", "mean"),
    ).round(3)

headline(us)

## 6. The full report

`write_report` produces `REPORT.md` with the banner, caveats, scenario provenance (judgment parameters listed explicitly), loadings, results with bootstrap intervals against the benchmark, and the variant comparison; PNG plots are added when matplotlib is installed.

In [ ]:
write_report(ai, "out_spy/ai_bubble")
write_report(us, "out_spy/us_bankruptcy")

## Limits

Bars are daily; finer bars are not disaggregated. The us_bankruptcy parameters flagged JUDGMENT in the report are assumptions with a stated rationale, not estimates. Past crises do not repeat. Not financial advice.